In [10]:
import json
import os
from typing import Dict, Tuple, Set
import pandas as pd


def load_json(path: str) -> Dict:
    if not os.path.exists(path):
        raise FileNotFoundError(f"Arquivo não encontrado: {path}")
    with open(path, "r", encoding="utf-8") as f:
        return json.load(f)


def canonicalize_link(link: str) -> str:
    """
    Transforma 'to-mt' e 'mt-to' no mesmo identificador.
    Mantém 'mt-ro' como 'mt-ro' (ordenando alfabeticamente os lados).
    """
    if "-" not in link:
        return link
    a, b = link.split("-", 1)
    return "-".join(sorted([a, b]))


def parse_lstm_gru(data: Dict, canonicalize_links: bool = False) -> Dict[str, Dict[str, float]]:
    """
    Retorna:
      { 'mt-ro': {'LSTM': nrmse, 'GRU': nrmse}, ... }
    """
    results: Dict[str, Dict[str, float]] = {}

    for key, metrics in data.items():
        if not isinstance(metrics, dict) or "NRMSE" not in metrics:
            continue

        parts = key.split("_")
        if len(parts) < 2:
            continue

        model = parts[-1]  # LSTM ou GRU
        if model not in {"LSTM", "GRU"}:
            continue

        link = parts[0]  # ex: mt-ro
        if canonicalize_links:
            link = canonicalize_link(link)

        results.setdefault(link, {})[model] = float(metrics["NRMSE"])

    return results


def parse_proposal(data: Dict, canonicalize_links: bool = False) -> Dict[str, float]:
    """
    Retorna:
      { 'mt-ro': nrmse, ... }
    """
    results: Dict[str, float] = {}

    for key, metrics in data.items():
        if not isinstance(metrics, dict) or "NRMSE" not in metrics:
            continue

        link = key.split("_")[0]  # ex: mt-ro
        if canonicalize_links:
            link = canonicalize_link(link)

        results[link] = float(metrics["NRMSE"])

    return results


def compare_nrmses(lstm_gru_path: str, proposal_path: str, canonicalize_links: bool = False) -> pd.DataFrame:
    lstm_gru_data = load_json(lstm_gru_path)
    proposal_data = load_json(proposal_path)

    lstm_gru = parse_lstm_gru(lstm_gru_data, canonicalize_links=canonicalize_links)
    proposal = parse_proposal(proposal_data, canonicalize_links=canonicalize_links)

    # links completos no arquivo 1 (tem LSTM e GRU)
    complete_lstm_gru: Set[str] = {k for k, v in lstm_gru.items() if "LSTM" in v and "GRU" in v}
    proposal_links: Set[str] = set(proposal.keys())

    common: Set[str] = complete_lstm_gru & proposal_links

    # ---- Diagnóstico ----
    print("=== Diagnóstico ===")
    print(f"Arquivo 1: links totais (com algum modelo): {len(lstm_gru)}")
    print(f"Arquivo 1: links completos (LSTM+GRU):     {len(complete_lstm_gru)}")
    print(f"Arquivo 2 (proposal): links:               {len(proposal_links)}")
    print(f"Links em comum (LSTM+GRU+proposal):        {len(common)}")

    if len(common) == 0:
        # Mostra alguns exemplos do que está faltando (pra não imprimir infinito)
        only_in_lstm_gru = sorted(list(complete_lstm_gru - proposal_links))[:25]
        only_in_proposal = sorted(list(proposal_links - complete_lstm_gru))[:25]
        print("\nExemplos só no arquivo 1 (completos LSTM+GRU), não estão no proposal:")
        print(only_in_lstm_gru)
        print("\nExemplos só no proposal, não estão completos no arquivo 1:")
        print(only_in_proposal)

    # ---- Montagem da tabela ----
    rows = []
    for link in sorted(common):
        rows.append({
            "link": link,
            "NRMSE_LSTM": lstm_gru[link]["LSTM"],
            "NRMSE_GRU": lstm_gru[link]["GRU"],
            "NRMSE_PROPOSAL": proposal[link],
            "Δ_PROPOSAL_vs_LSTM": proposal[link] - lstm_gru[link]["LSTM"],
            "Δ_PROPOSAL_vs_GRU": proposal[link] - lstm_gru[link]["GRU"],
        })

    return pd.DataFrame(rows)


if __name__ == "__main__":
    proposal_file = "../results/bbr/evaluation.json"
    lstm_gru_file = "cubic/evaluation.json"
    
    # tente primeiro False; se suspeitar de links invertidos (to-mt vs mt-to), use True
    df = compare_nrmses(lstm_gru_file, proposal_file, canonicalize_links=True)

    print("\n=== Comparação ===")
    print(df if not df.empty else "Sem links correspondentes nos 3 modelos.")


=== Diagnóstico ===
Arquivo 1: links totais (com algum modelo): 74
Arquivo 1: links completos (LSTM+GRU):     74
Arquivo 2 (proposal): links:               112
Links em comum (LSTM+GRU+proposal):        74

=== Comparação ===
     link  NRMSE_LSTM  NRMSE_GRU  NRMSE_PROPOSAL  Δ_PROPOSAL_vs_LSTM  \
0   ac-ap    0.969255   0.884461        0.212221           -0.757035   
1   ac-ba    0.886660   0.951252        0.093298           -0.793362   
2   ac-go    0.678822   0.760891        0.116589           -0.562233   
3   ac-ms    0.904161   0.833762        0.093059           -0.811102   
4   ac-mt    0.528511   0.091551        0.092883           -0.435629   
..    ...         ...        ...             ...                 ...   
69  pi-pr    0.036339   0.061256        0.037201            0.000861   
70  pi-ro    0.056880   0.191034        0.058053            0.001173   
71  pi-rs    0.061742   0.061672        0.070147            0.008405   
72  rs-to    0.010950   0.010709        0.010874      

In [11]:
df.to_csv("oi.csv")